In [1]:
import geogridfusion
import pvdeg
import pvlib

In [6]:
# downloads container from docker
geogridfusion.run_container(accept_docker=None)

=== GeoGridFusion will use Docker ===
- Action: PULL & RUN a container image
- Image:  postgis/postgis:14-3.5
- Source: Docker Hub
- Port:   127.0.0.1:5433 -> container:5432
- Data:   named volume "pgdata"

This will download data from the internet and start a background service.


RuntimeError: Refusing to pull/run a Docker image without explicit consent.
Re-run with accept_docker=True.

In [3]:
fusion = geogridfusion.geogridfusionStore()
fusion.connect()

PostgreSQL connection established after 0.02 seconds.


In [4]:
sw, sm = pvlib.iotools.get_solrad(station="abq", start="2022-01-01", end="2022-01-05")

In [5]:
fusion.store_single(
    weather_df=pvdeg.weather.map_weather(sw),
    meta=pvdeg.weather.map_meta(sm),
    source_name="solrad",
    tmy=False,
)

duplicate file detected, skipping insert
metadata of duplicate file {'station': 'abq', 'filenames': ['abq/2022/abq22001.dat', 'abq/2022/abq22002.dat', 'abq/2022/abq22003.dat', 'abq/2022/abq22004.dat', 'abq/2022/abq22005.dat'], 'station_name': 'Albuquerque', 'latitude': 35.03796, 'longitude': -106.62211, 'altitude': 1617.0, 'TZ': -7}


In [ ]:
client = pvdeg.geospatial.start_dask()

coords = [(45 + i, -115 + k) for i in range(5) for k in range(5)]

geo_weather, geo_meta, failed = pvdeg.weather.weather_distributed(
    database="PVGIS", coords=coords
)

client.close()

In [ ]:
weather, meta = pvdeg.weather.get(database="PVGIS", id=(45, -115))

In [ ]:
geogridfusion.store_single(
    conn=conn, weather_df=weather, meta=meta, tmy=True, source_res="pvgis"
)

When we run the cell below, we see that we will get a collsion from the data inserted above.

We want to rewrite this function so it can be done async or with multiprocessing.

In [ ]:
for i in range(25):
    w = geo_weather.isel(gid=i).drop_vars(("gid",)).to_pandas()
    m = geo_meta.iloc[i].to_dict()

    geogridfusion.store_single(
        conn=conn, weather_df=w, meta=m, tmy=True, source_name="pvgis"
    )

We can easily get one output from what we have written so far. We need to be able to read MANY into dataset form.

In [ ]:
geogridfusion.sources(conn)

In [ ]:
lw, lm = geogridfusion.load_single(
    conn=conn,
    latitude=sm["latitude"] + 10,
    longitude=sm["longitude"],
    source_name="solrad",
)

lw.shape

In [ ]:
gw, gm = geogridfusion.load_many(
    conn=conn,
    source_name="pvgis",
    spatial_search=True,
    spatial_search_distance_floor=140 * 1000,
    spatial_search_latitude0=45,
    spatial_search_longitude0=-114,
)

In [ ]:
gm

In [ ]:
gm